<a href="https://colab.research.google.com/github/yosie111/Shulchan_Aruch_RAG_1/blob/main/SA_RAG_Stage3_Reranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shulchan Aruch RAG — Stage 3: Reranker

## מיקום בארכיטקטורה:
```
שאלה → Vector Search (top-20) → Reranker (top-5) → LLM → תשובה
```

## למה reranker?
Vector search (bge-m3) הגיע ל-88% Recall@5 ונכשל ב-12 שאלות.
Reranker הוא cross-encoder שבודק כל צמד (שאלה, chunk) בנפרד —
מדויק הרבה יותר מ-bi-encoder, אבל איטי (לכן רק על top-20).

## קלט:
- `shulchan_aruch_v2.jsonl` — chunks
- `על_שוע_חלק_א.xlsx` — 102 שאלות evaluation

## פלט:
- טבלת השוואה: vector בלבד vs vector+reranker
- שיפור per-שאלה
- קונפיגורציה מומלצת ל-Stage 3 (Generation)

## הוראות:
1. הרץ תאים 1-5 ברצף


In [1]:
# ============================================================
# תא 1 — התקנה + ייבוא
# ============================================================
!pip install -q sentence-transformers chromadb openpyxl rank-bm25

import json
import re
import os
import time
import numpy as np
from pathlib import Path

import openpyxl
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('✅ תא 1')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 147.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 175.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cu

In [2]:
# ============================================================
# תא 2 — טעינת נתונים + retriever
# ============================================================

# --- A. JSONL ---
JSONL_FILE = None
for p in sorted(Path('.').glob('*.jsonl')):
    JSONL_FILE = str(p); break
if JSONL_FILE is None and IN_COLAB:
    print('העלה JSONL:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.jsonl'): JSONL_FILE = n; break
if not JSONL_FILE: raise FileNotFoundError('JSONL not found')

chunks = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): chunks.append(json.loads(line))
content_chunks = [c for c in chunks if c.get('metadata',{}).get('doc_type') != 'hilchot_index']
print(f'📄 {len(content_chunks)} chunks')

# --- B. Excel ---
EVAL_FILE = None
for p in sorted(Path('.').glob('*.xlsx')):
    EVAL_FILE = str(p); break
if EVAL_FILE is None and IN_COLAB:
    print('העלה Excel:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.xlsx'): EVAL_FILE = n; break

GEMATRIA = {'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90}
def heb2num(s):
    return sum(GEMATRIA.get(c,0) for c in re.sub(r'["\'\'  ׳״]','',s or ''))

eval_questions = []
if EVAL_FILE:
    wb = openpyxl.load_workbook(EVAL_FILE); ws = wb.active
    for row in range(2, ws.max_row + 1):
        q = ws.cell(row,1).value
        if not q: continue
        src = ws.cell(row,3).value or ''
        sm = re.search(r'סימן\s+([א-ת]+)', src)
        sem = re.search(r'סעיף\s+([א-ת]+)', src)
        eval_questions.append({
            'question': q,
            'answer': ws.cell(row,2).value or '',
            'source': src,
            'gold_siman': heb2num(sm.group(1)) if sm else 0,
            'gold_seif': heb2num(sem.group(1)) if sem else 0,
        })
    print(f'📝 {len(eval_questions)} שאלות')

# --- C. Gold mapping ---
def find_gold_chunk(q, chs):
    gs, gse = q['gold_siman'], q['gold_seif']
    for c in chs:
        h = c.get('metadata',{}).get('hierarchy',{})
        if h.get('level_1_num') == gs:
            if gse in h.get('level_2_nums',[]) or gse == 0:
                return c['doc_id']
    for c in chs:
        if c.get('metadata',{}).get('hierarchy',{}).get('level_1_num') == gs:
            return c['doc_id']
    return None

for q in eval_questions:
    q['gold_chunk_id'] = find_gold_chunk(q, content_chunks)
valid_questions = [q for q in eval_questions if q['gold_chunk_id']]
print(f'📌 {len(valid_questions)} שאלות עם gold mapping')

# --- D. Bi-encoder (bge-m3) ---
print('Loading bge-m3...')
bi_encoder = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

chunk_ids = [c['doc_id'] for c in content_chunks]
chunk_texts_emb = [c['content']['text_for_embedding'] for c in content_chunks]
chunk_texts_llm = [c['content']['text_for_llm'] for c in content_chunks]
chunk_metas = [{'siman': c['metadata']['hierarchy']['level_1_num']} for c in content_chunks]

# chunk_id → text_for_llm
id_to_text = {c['doc_id']: c['content']['text_for_llm'] for c in content_chunks}

print('Encoding chunks...')
doc_vectors = bi_encoder.encode(chunk_texts_emb, show_progress_bar=True, normalize_embeddings=True)

chroma = chromadb.Client()
try: chroma.delete_collection('sa_rerank')
except: pass
collection = chroma.create_collection('sa_rerank', metadata={'hnsw:space': 'cosine'})
collection.add(
    ids=chunk_ids,
    embeddings=[v.tolist() for v in doc_vectors],
    documents=chunk_texts_llm,
    metadatas=chunk_metas,
)
print(f'✅ Vector DB: {collection.count()} chunks')


העלה JSONL:


Saving shulchan_aruch_v2 (11).jsonl to shulchan_aruch_v2 (11).jsonl
📄 1355 chunks
העלה Excel:


Saving על שוע    חלק א.xlsx to על שוע    חלק א.xlsx
📝 102 שאלות
📌 102 שאלות עם gold mapping
Loading bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Encoding chunks...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

✅ Vector DB: 1355 chunks


In [3]:
from rank_bm25 import BM25Okapi

print('Building BM25 index...')
chunk_texts_bm25 = [c['content']['text_for_bm25'] for c in content_chunks]
tokenized_corpus = [text.split() for text in chunk_texts_bm25]
bm25_index = BM25Okapi(tokenized_corpus)
print('✅ BM25 Ready')

Building BM25 index...
✅ BM25 Ready


In [4]:
# ============================================================
# תא 3 — טעינת Reranker models
# ============================================================
from collections import defaultdict
import numpy as np

RERANKERS = {
    'bge-reranker-v2-m3': 'BAAI/bge-reranker-v2-m3',
    'mxbai-rerank-base': 'mixedbread-ai/mxbai-rerank-base-v1',
}

reranker_models = {}
for name, model_path in RERANKERS.items():
    print(f'Loading {name}...')
    t0 = time.time()
    reranker_models[name] = CrossEncoder(model_path, device=DEVICE)
    print(f'  Loaded in {time.time()-t0:.1f}s')


# === Retrieval pipeline functions ===

def retrieve_top_n(question, n=20):
    """bi-encoder retrieval → top-N chunk_ids + texts"""
    qvec = bi_encoder.encode([question], normalize_embeddings=True)
    res = collection.query(
        query_embeddings=[qvec[0].tolist()],
        n_results=n,
    )
    return list(zip(res['ids'][0], res['documents'][0]))


def retrieve_top_n(question, n=22, vector_weight=0.7):
    """Hybrid Retrieval: Vector (bge-m3) + Keyword (BM25) using RRF"""
    rrf_k = 60

    # 1. Vector Retrieval
    qvec = bi_encoder.encode([question], normalize_embeddings=True)
    vec_res = collection.query(
        query_embeddings=[qvec[0].tolist()],
        n_results=n
    )
    vec_ids = vec_res['ids'][0]

    # 2. BM25 Retrieval
    query_tokens = question.split()
    bm25_scores = bm25_index.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:n]
    bm25_ids = [chunk_ids[i] for i in bm25_top_indices]

    # 3. RRF Fusion
    rrf_scores = defaultdict(float)
    for rank, cid in enumerate(vec_ids):
        rrf_scores[cid] += vector_weight / (rrf_k + rank + 1)
    for rank, cid in enumerate(bm25_ids):
        rrf_scores[cid] += (1 - vector_weight) / (rrf_k + rank + 1)

    # מיון לפי הציון המשוקלל
    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:n]

    return [(cid, id_to_text[cid]) for cid in sorted_ids]

def rerank(question, candidates, reranker_model, top_k=5):
    """cross-encoder reranking: מדרג מחדש (שאלה, chunk) pairs"""
    if not candidates:
        return []
    pairs = [(question, doc_text) for _, doc_text in candidates]
    scores = reranker_model.predict(pairs)
    scored = list(zip(candidates, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [(cid, text, float(score)) for (cid, text), score in scored[:top_k]]


# === Metric functions ===
K_VALUES = [1, 3, 5, 10]

def recall_at_k(ids, gold, k):
    return 1.0 if gold in ids[:k] else 0.0

def mrr(ids, gold):
    for i, rid in enumerate(ids):
        if rid == gold: return 1.0 / (i + 1)
    return 0.0

def compute_metrics(all_retrieved, all_gold):
    n = len(all_retrieved)
    m = {}
    for k in K_VALUES:
        m[f'Recall@{k}'] = np.mean([recall_at_k(r, g, k) for r, g in zip(all_retrieved, all_gold)])
    m['MRR'] = np.mean([mrr(r, g) for r, g in zip(all_retrieved, all_gold)])
    return m


print(f'✅ תא 3 — {len(reranker_models)} rerankers מוכנים')


Loading bge-reranker-v2-m3...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

  Loaded in 8.9s
✅ תא 3 — 2 rerankers מוכנים


In [8]:
# ============================================================
# תא 4 — Benchmark: vector בלבד vs vector+reranker
# ============================================================

TOP_N_RETRIEVE = 22  # שליפה ראשונית
TOP_K_RERANK = 10    # אחרי rerank

all_results = {}  # method → {'retrieved': [...], 'gold': [...]}

# --- A. Vector only (baseline) ---
print('Running vector baseline...')
t0 = time.time()
vec_retrieved = []
gold_ids = []
for q in valid_questions:
    candidates = retrieve_top_n(q['question'], n=TOP_N_RETRIEVE)
    vec_retrieved.append([cid for cid, _ in candidates])
    gold_ids.append(q['gold_chunk_id'])
all_results['bge-m3 (vector)'] = (vec_retrieved, gold_ids)
print(f'  Done in {time.time()-t0:.1f}s')

# --- B. Vector + Reranker (per model) ---
for rr_name, rr_model in reranker_models.items():
    print(f'\nRunning {rr_name}...')
    t0 = time.time()
    rr_retrieved = []
    for i, q in enumerate(valid_questions):
        candidates = retrieve_top_n(q['question'], n=TOP_N_RETRIEVE)
        reranked = rerank(q['question'], candidates, rr_model, top_k=TOP_K_RERANK)
        rr_retrieved.append([cid for cid, _, _ in reranked])
        if (i+1) % 25 == 0:
            print(f'  [{i+1}/{len(valid_questions)}]')
    all_results[f'bge-m3 + {rr_name}'] = (rr_retrieved, gold_ids)
    print(f'  Done in {time.time()-t0:.1f}s')

# --- C. Compute metrics ---
print(f'\n{"="*80}')
print(f'{"RERANKER BENCHMARK":^80}')
print(f'{"="*80}')
print()

header = f'{"Method":<35}'
for k in K_VALUES:
    header += f'{"Recall@"+str(k):>10}'
header += f'{"MRR":>10}'
print(header)
print('-' * len(header))

all_metrics = {}
for method, (retrieved, gold) in all_results.items():
    m = compute_metrics(retrieved, gold)
    all_metrics[method] = m
    row = f'{method:<35}'
    for k in K_VALUES:
        row += f'{m[f"Recall@{k}"]:>10.3f}'
    row += f'{m["MRR"]:>10.3f}'
    print(row)

# --- D. Improvement analysis ---
print(f'\n--- שיפור מעל baseline ---')
baseline_r5 = all_metrics['bge-m3 (vector)']['Recall@5']
baseline_mrr = all_metrics['bge-m3 (vector)']['MRR']
for method, m in all_metrics.items():
    if method == 'bge-m3 (vector)': continue
    dr5 = m['Recall@5'] - baseline_r5
    dmrr = m['MRR'] - baseline_mrr
    print(f'  {method}: Recall@5 {dr5:+.3f}, MRR {dmrr:+.3f}')

# Best
best_method = max(all_metrics, key=lambda x: all_metrics[x]['Recall@5'])
best_r5 = all_metrics[best_method]['Recall@5']
print(f'\n🏆 Best: {best_method} (Recall@5={best_r5:.3f})')

# --- E. Per-question comparison: fixed by reranker ---
print(f'\n--- שאלות שהreranker תיקן ---')
best_rr_key = best_method
best_rr_retrieved = all_results[best_rr_key][0]

fixed = []
broke = []
for i, q in enumerate(valid_questions):
    gold = q['gold_chunk_id']
    vec_hit = gold in vec_retrieved[i][:5]
    rr_hit = gold in best_rr_retrieved[i][:5]
    if not vec_hit and rr_hit:
        fixed.append((q, vec_retrieved[i][:3], best_rr_retrieved[i][:3]))
    elif vec_hit and not rr_hit:
        broke.append((q, vec_retrieved[i][:3], best_rr_retrieved[i][:3]))

print(f'  תוקנו: {len(fixed)}')
for q, old, new in fixed[:5]:
    print(f'    Q: {q["question"][:55]}')
    print(f'    vector: {old} → reranked: {new}')

print(f'  נשברו: {len(broke)}')
for q, old, new in broke[:5]:
    print(f'    Q: {q["question"][:55]}')
    print(f'    vector: {old} → reranked: {new}')

print(f'\n✅ תא 4')


Running vector baseline...
  Done in 0.7s

Running bge-reranker-v2-m3...
  [25/102]
  [50/102]
  [75/102]
  [100/102]
  Done in 16.4s

Running mxbai-rerank-base...
  [25/102]
  [50/102]
  [75/102]
  [100/102]
  Done in 6.6s

                               RERANKER BENCHMARK                               

Method                               Recall@1  Recall@3  Recall@5 Recall@10       MRR
-------------------------------------------------------------------------------------
bge-m3 (vector)                         0.608     0.745     0.794     0.853     0.689
bge-m3 + bge-reranker-v2-m3             0.716     0.873     0.873     0.892     0.796
bge-m3 + mxbai-rerank-base              0.500     0.676     0.716     0.814     0.605

--- שיפור מעל baseline ---
  bge-m3 + bge-reranker-v2-m3: Recall@5 +0.078, MRR +0.107
  bge-m3 + mxbai-rerank-base: Recall@5 -0.078, MRR -0.084

🏆 Best: bge-m3 + bge-reranker-v2-m3 (Recall@5=0.873)

--- שאלות שהreranker תיקן ---
  תוקנו: 9
    Q: באיזו יד נוטל א

In [9]:
# ============================================================
# תא 5 — שמירה + הורדה
# ============================================================

OUTPUT_DIR = '/content/reranker_output' if IN_COLAB else 'reranker_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- A. JSON — כל התוצאות ---
output = {
    'config': {
        'bi_encoder': 'BAAI/bge-m3',
        'rerankers': list(RERANKERS.keys()),
        'top_n_retrieve': TOP_N_RETRIEVE,
        'top_k_rerank': TOP_K_RERANK,
        'n_questions': len(valid_questions),
    },
    'metrics': {method: {k: round(v,4) if isinstance(v,float) else v
                         for k,v in m.items()}
                for method, m in all_metrics.items()},
    'best_method': best_method,
    'fixed_by_reranker': len(fixed),
    'broke_by_reranker': len(broke),
}

json_path = os.path.join(OUTPUT_DIR, 'reranker_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f'📄 {json_path}')

# --- B. CSV ---
csv_path = os.path.join(OUTPUT_DIR, 'reranker_comparison.csv')
with open(csv_path, 'w', encoding='utf-8-sig') as f:
    header = 'Method'
    for k in K_VALUES: header += f',Recall@{k}'
    header += ',MRR'
    f.write(header + '\n')
    for method in sorted(all_metrics, key=lambda x: -all_metrics[x]['Recall@5']):
        m = all_metrics[method]
        row = method
        for k in K_VALUES: row += f',{m[f"Recall@{k}"]:.4f}'
        row += f',{m["MRR"]:.4f}'
        f.write(row + '\n')
print(f'📄 {csv_path}')

# --- C. Per-question detail ---
detail_rows = []
for i, q in enumerate(valid_questions):
    gold = q['gold_chunk_id']
    detail_rows.append({
        'question': q['question'],
        'gold': gold,
        'source': q.get('source',''),
        'vector_top5': vec_retrieved[i][:5],
        'vector_hit': gold in vec_retrieved[i][:5],
        'reranked_top5': best_rr_retrieved[i][:5],
        'reranked_hit': gold in best_rr_retrieved[i][:5],
    })

detail_path = os.path.join(OUTPUT_DIR, 'per_question_detail.json')
with open(detail_path, 'w', encoding='utf-8') as f:
    json.dump(detail_rows, f, ensure_ascii=False, indent=2)
print(f'📄 {detail_path}')

# --- D. Summary ---
summary = []
summary.append('Reranker Benchmark Summary')
summary.append('=' * 50)
summary.append(f'Best: {best_method}')
summary.append(f'Recall@5: {baseline_r5:.3f} → {best_r5:.3f} (Δ={best_r5-baseline_r5:+.3f})')
summary.append(f'Fixed: {len(fixed)} questions | Broke: {len(broke)} questions')
summary.append('')
summary.append('Recommendation for Stage 3 (Generation):')
if best_r5 > baseline_r5 + 0.02:
    summary.append(f'  USE reranker ({best_method}) — significant improvement')
elif best_r5 >= baseline_r5:
    summary.append(f'  OPTIONAL reranker — marginal improvement, adds latency')
else:
    summary.append(f'  SKIP reranker — no improvement over vector baseline')

summary_path = os.path.join(OUTPUT_DIR, 'reranker_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary))
print(f'📄 {summary_path}')

# Print summary
print(f'\n{"="*50}')
for line in summary: print(f'  {line}')
print(f'{"="*50}')

# --- E. Download ---
print(f'\n📂 {os.path.abspath(OUTPUT_DIR)}/')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f'  📄 {fname} ({os.path.getsize(fpath):,} bytes)')

if IN_COLAB:
    print('\n⬇️  מוריד...')
    from google.colab import files as colab_files
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        colab_files.download(os.path.join(OUTPUT_DIR, fname))
    print('✅ הורדו!')
else:
    print(f'\n✅ סיום')


📄 /content/reranker_output/reranker_results.json
📄 /content/reranker_output/reranker_comparison.csv
📄 /content/reranker_output/per_question_detail.json
📄 /content/reranker_output/reranker_summary.txt

  Reranker Benchmark Summary
  Best: bge-m3 + bge-reranker-v2-m3
  Recall@5: 0.794 → 0.873 (Δ=+0.078)
  Fixed: 9 questions | Broke: 1 questions
  
  Recommendation for Stage 3 (Generation):
    USE reranker (bge-m3 + bge-reranker-v2-m3) — significant improvement

📂 /content/reranker_output/
  📄 per_question_detail.json (52,525 bytes)
  📄 reranker_comparison.csv (227 bytes)
  📄 reranker_results.json (817 bytes)
  📄 reranker_summary.txt (304 bytes)

⬇️  מוריד...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ הורדו!
